# <font color="#418FDE" size="6.5" uppercase>**Neural Network Foundations**</font>

>Last update: 20260327.
    
By the end of this Lecture, you will be able to:
- Explain the perceptron model, its biological inspiration, and the roles of weights, biases, and activation functions. 
- Describe the structure of multi-layer perceptrons and distinguish between shallow and deep neural networks. 
- Interpret the significance of the universal approximation theorem for modeling complex CE relationships. 


## **1. Perceptron Fundamentals**

### **1.1. Biological Inspiration**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_A/image_01_01.jpg?v=1774622843" width="250">



>* Neurons combine inputs and trigger outputs.
>* Simple units together enable complex behavior.

>* Inputs have different levels of influence.
>* Perceptrons combine signals by importance.

>* Perceptrons greatly simplify real brain processes.
>* Simple units combine to learn patterns.



In [ ]:
#@title Python Code - Biological Inspiration

# Perceptrons borrow ideas from biological neurons.
# Inputs combine before producing one output.
# This example uses concrete strength data.

# Download the concrete dataset file.
!wget -q -O ./Concrete_Data.xls https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls

# Import simple tools for data handling.
import numpy as np
import pandas as pd

# Set a seed for repeatable choices.
np.random.seed(7)

# Load the spreadsheet into pandas.
df = pd.read_excel("./Concrete_Data.xls")

# Keep only needed columns here.
feature_names = [
    "Cement (component 1)(kg in a m^3 mixture)",
    "Water  (component 4)(kg in a m^3 mixture)",
    "Age (day)",
    "Concrete compressive strength(MPa, megapascals) "
]

# Select a small teaching table.
data = df[feature_names].copy()

# Rename columns for easier reading.
data.columns = ["cement", "water", "age", "strength"]

# Create a simple quality label.
data["good_concrete"] = (data["strength"] >= 40).astype(int)

# Choose one example row safely.
row = data.iloc[0]

# Build three perceptron input values.
inputs = np.array([
    row["cement"] / data["cement"].max(),
    row["water"] / data["water"].max(),
    row["age"] / data["age"].max()
])

# Assign weights like connection strengths.
weights = np.array([0.8, -0.7, 0.5])

# Add a bias like firing tendency.
bias = -0.2

# Compute the weighted sum signal.
weighted_sum = np.sum(inputs * weights) + bias

# Apply a threshold activation rule.
prediction = int(weighted_sum >= 0)

# Compare with the dataset label.
actual = int(row["good_concrete"])

# Explain the neuron analogy briefly.
print("Biological idea: many signals combine into one decision.")
print("Dataset row index:", int(row.name))
print("Normalized inputs:", np.round(inputs, 3))
print("Weights:", weights)
print("Bias:", bias)
print("Weighted sum:", round(float(weighted_sum), 3))
print("Activation output:", prediction)
print("Actual quality label:", actual)
print("1 means stronger concrete, 0 means weaker concrete.")



### **1.2. Rosenblatt Perceptron Model**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_A/image_01_02.jpg?v=1774622874" width="250">



>* Early neuron-inspired model for simple decisions
>* Classifies inputs into two categories

>* Perceptron learns by adjusting from mistakes.
>* Training data shifts boundaries for classification.

>* Perceptrons handle simple linear classification tasks.
>* Complex nonlinear problems need advanced networks.



In [ ]:
#@title Python Code - Rosenblatt Perceptron Model

# This notebook shows a simple perceptron.
# We classify concrete strength into classes.
# The example uses Rosenblatt learning ideas.

# Download the concrete dataset file.
!wget -q -O ./Concrete_Data.xls https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls

# Import beginner friendly data tools.
import numpy as np
import pandas as pd

# Set a deterministic random seed.
np.random.seed(7)

# Load the spreadsheet into pandas.
df = pd.read_excel("./Concrete_Data.xls")

# Keep only a small preview size.
row_count = int(df.shape[0])
col_count = int(df.shape[1])

# Separate inputs and target column.
X_raw = df.iloc[:, :-1].to_numpy(dtype=float)
y_strength = df.iloc[:, -1].to_numpy(dtype=float)

# Create binary labels from median strength.
threshold = float(np.median(y_strength))
y = (y_strength >= threshold).astype(int)

# Check basic dataset dimensions safely.
if row_count < 20 or col_count < 2:
    raise ValueError("Dataset shape is unexpectedly small.")

# Standardize each input feature simply.
means = X_raw.mean(axis=0)
stds = X_raw.std(axis=0)
stds[stds == 0.0] = 1.0
X = (X_raw - means) / stds

# Shuffle indices for train test split.
indices = np.arange(X.shape[0])
np.random.shuffle(indices)

# Build a simple eighty twenty split.
split = int(0.8 * len(indices))
train_idx = indices[:split]
test_idx = indices[split:]

# Create train and test arrays.
X_train = X[train_idx]
y_train = y[train_idx]
X_test = X[test_idx]
y_test = y[test_idx]

# Convert labels for perceptron updates.
y_train_pm = np.where(y_train == 1, 1, -1)
y_test_pm = np.where(y_test == 1, 1, -1)

# Start weights and bias at zero.
weights = np.zeros(X_train.shape[1], dtype=float)
bias = 0.0

# Choose a small learning rate.
learning_rate = 0.05
epochs = 12

# Train a Rosenblatt style perceptron.
for epoch in range(epochs):
    mistakes = 0
    for i in range(X_train.shape[0]):
        score = np.dot(X_train[i], weights) + bias
        pred = 1 if score >= 0.0 else -1
        if pred != y_train_pm[i]:
            weights = weights + learning_rate * y_train_pm[i] * X_train[i]
            bias = bias + learning_rate * y_train_pm[i]
            mistakes = mistakes + 1

# Predict classes on train data.
train_scores = X_train @ weights + bias
train_pred = (train_scores >= 0.0).astype(int)

# Predict classes on test data.
test_scores = X_test @ weights + bias
test_pred = (test_scores >= 0.0).astype(int)

# Compute simple accuracy values.
train_acc = float((train_pred == y_train).mean())
test_acc = float((test_pred == y_test).mean())

# Show one concrete sample decision.
example_index = 0
example_score = float(test_scores[example_index])
example_true = int(y_test[example_index])
example_pred = int(test_pred[example_index])

# Print a short teaching summary.
print("Dataset rows:", row_count)
print("Input features:", X_train.shape[1])
print("Strength threshold:", round(threshold, 2), "MPa")
print("Train accuracy:", round(train_acc, 3))
print("Test accuracy:", round(test_acc, 3))
print("Bias term:", round(float(bias), 3))
print("First weight:", round(float(weights[0]), 3))
print("Example score:", round(example_score, 3))
print("Example true class:", example_true)
print("Example predicted class:", example_pred)
print("Activation rule: score >= 0 means high strength.")



### **1.3. Weights and Biases**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_A/image_01_03.jpg?v=1774622908" width="250">



>* Weights scale each input's importance.
>* Bias shifts the activation threshold.

>* Weights reflect feature importance in decisions.
>* Bias shifts the decision boundary position.

>* Training adjusts weights and biases from data.
>* Weights count evidence; biases set activation threshold.



In [ ]:
#@title Python Code - Weights and Biases

# This script shows neuron weights clearly.
# Air quality data supports engineering context.
# Bias shifts the alert threshold.

# Import simple data tools.
import numpy as np
import pandas as pd

# Download the dataset file.
!wget -q -O ./AirQualityUCI.zip https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip
# Unzip the downloaded dataset.
!unzip -o -q ./AirQualityUCI.zip

# Set a fixed random seed.
np.random.seed(7)
# Load the air quality table.
df = pd.read_csv(
    "AirQualityUCI.csv", sep=";", decimal="," 
)

# Keep only useful columns.
cols = ["CO(GT)", "NO2(GT)", "RH"]
# Select a small clean table.
data = df[cols].copy()
# Replace missing value markers.
data = data.replace(-200, np.nan)
# Remove rows with missing values.
data = data.dropna()

# Stop early if data failed.
if len(data) == 0:
    raise ValueError("Dataset cleaning removed all rows.")

# Take one realistic sample row.
sample = data.iloc[10]
# Build the input vector.
x = np.array([
    sample["CO(GT)"], sample["NO2(GT)"], sample["RH"]
], dtype=float)

# Scale features for easier comparison.
scales = np.array([10.0, 200.0, 100.0], dtype=float)
# Create normalized inputs.
x_scaled = x / scales
# Check the input shape.
if x_scaled.shape != (3,):
    raise ValueError("Input vector has wrong shape.")

# Define a simple sigmoid function.
def sigmoid(z):

    return 1.0 / (1.0 + np.exp(-z))

# Choose baseline neuron weights.
weights_a = np.array([1.8, 1.2, -0.8], dtype=float)
# Choose a baseline neuron bias.
bias_a = -0.4
# Compute the weighted sum.
z_a = np.dot(weights_a, x_scaled) + bias_a
# Convert sum into alert score.
y_a = sigmoid(z_a)

# Increase pollution importance weights.
weights_b = np.array([3.0, 2.0, -0.8], dtype=float)
# Keep the same bias first.
bias_b = -0.4
# Compute the new weighted sum.
z_b = np.dot(weights_b, x_scaled) + bias_b
# Convert sum into alert score.
y_b = sigmoid(z_b)

# Keep weights but raise bias.
weights_c = weights_b.copy()
# Make activation harder overall.
bias_c = -1.2
# Compute the shifted weighted sum.
z_c = np.dot(weights_c, x_scaled) + bias_c
# Convert sum into alert score.
y_c = sigmoid(z_c)

# Print one compact teaching summary.
print("Sample inputs:", np.round(x, 2))
print("Scaled inputs:", np.round(x_scaled, 3))
print("Case A score:", round(float(y_a), 3))
print("Case B score:", round(float(y_b), 3))
print("Case C score:", round(float(y_c), 3))
print("A uses moderate weights and bias.")
print("B raises pollution weights, so score increases.")
print("C lowers bias, so activation becomes harder.")
print("Alert if score exceeds 0.5:", y_c > 0.5)



## **2. Network Structures**

### **2.1. Activation Functions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_A/image_02_01.jpg?v=1774622942" width="250">



>* Activation functions add essential nonlinearity to networks.
>* They enable complex patterns beyond linear behavior.

>* Activation choice affects learning and training efficiency.
>* They highlight thresholds or preserve gradual changes.

>* Hidden layers build increasingly abstract features.
>* Output activations vary; deeper networks capture complexity.



In [ ]:
#@title Python Code - Activation Functions

# Activation functions shape neuron responses.
# This example uses air quality data.
# Civil engineering students compare common functions.

import zipfile
import numpy as np
import pandas as pd

# Set a seed for consistency.
np.random.seed(7)

# Download the air quality dataset.
!wget -q -O ./AirQualityUCI.zip https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip

# Extract the downloaded zip file.
with zipfile.ZipFile("./AirQualityUCI.zip", "r") as zf:
    zf.extractall("./air_quality_data")

# Load the main csv file.
file_path = "./air_quality_data/AirQualityUCI.csv"
df = pd.read_csv(file_path, sep=";", decimal=",")

# Remove empty trailing columns.
df = df.dropna(axis=1, how="all")

# Replace missing markers safely.
df = df.replace(-200, np.nan)

# Keep useful sensor columns only.
cols = ["CO(GT)", "PT08.S1(CO)", "NMHC(GT)", "C6H6(GT)"]
df = df[cols].copy()

# Drop rows missing selected values.
df = df.dropna().reset_index(drop=True)

# Stop early if data is missing.
if len(df) < 5:
    raise ValueError("Dataset has too few clean rows.")

# Build simple neuron inputs.
X = df[["PT08.S1(CO)", "NMHC(GT)", "C6H6(GT)"]].head(5).to_numpy()

# Standardize inputs for fair comparison.
means = X.mean(axis=0)
stds = X.std(axis=0) + 1e-8
X_scaled = (X - means) / stds

# Define weights and bias.
weights = np.array([0.8, -0.4, 0.6])
bias = -0.1

# Compute sample neuron outputs.
z = X_scaled @ weights + bias

# Define the step activation.
def step_function(values):

    return (values >= 0).astype(int)

# Define the sigmoid activation.
def sigmoid_function(values):

    return 1.0 / (1.0 + np.exp(-values))

# Define the tanh activation.
def tanh_function(values):

    return np.tanh(values)

# Define the ReLU activation.
def relu_function(values):

    return np.maximum(0, values)

# Apply each activation function.
step_out = step_function(z)
sigmoid_out = sigmoid_function(z)

tanh_out = tanh_function(z)
relu_out = relu_function(z)

# Print a short teaching summary.
print("Clean rows used:", len(df))
print("Neuron raw outputs:", np.round(z, 3))
print("Step outputs:", step_out)
print("Sigmoid outputs:", np.round(sigmoid_out, 3))

print("Tanh outputs:", np.round(tanh_out, 3))
print("ReLU outputs:", np.round(relu_out, 3))
print("Target example: air-quality risk scoring")
print("Step gives threshold decisions.")

print("Sigmoid gives probability-like values.")
print("Tanh keeps positive and negative influence.")
print("ReLU keeps strong positive signals.")



### **2.2. MLP Layer Structure**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_A/image_02_02.jpg?v=1774622976" width="250">



>* MLPs pass data through layered connections.
>* Hidden layers transform inputs into patterns.

>* Layers transform inputs through weights, biases, activations.
>* Deeper layers learn patterns for outputs.

>* Layer size controls learning capacity and complexity.
>* Inputs and hidden layers shape predictions.



In [ ]:
#@title Python Code - MLP Layer Structure

# This notebook shows a simple MLP structure.
# We predict normalized concrete strength values.
# One hidden layer demonstrates network organization.

# Install Excel reader if needed.
# !pip install xlrd

# Import basic learning tools.
import numpy as np
import pandas as pd

# Set a deterministic random seed.
np.random.seed(7)

# Download the concrete dataset.
!wget -q -O ./Concrete_Data.xls https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls

# Load the spreadsheet into pandas.
df = pd.read_excel("./Concrete_Data.xls")

# Show dataset size briefly.
print("Rows and columns:", df.shape)

# Convert table values to numbers.
data = df.to_numpy(dtype=float)
X = data[:, :-1]
y = data[:, -1:]

# Normalize inputs and target.
X_min = X.min(axis=0, keepdims=True)
X_max = X.max(axis=0, keepdims=True)

y_min = y.min(axis=0, keepdims=True)
y_max = y.max(axis=0, keepdims=True)

# Prevent division by zero safely.
X_range = X_max - X_min + 1e-8
y_range = y_max - y_min + 1e-8

# Create normalized learning arrays.
Xn = (X - X_min) / X_range
yn = (y - y_min) / y_range

# Split data into train and test.
count = Xn.shape[0]
indices = np.arange(count)
np.random.shuffle(indices)

# Use a small test portion.
train_end = int(0.8 * count)
train_ids = indices[:train_end]
test_ids = indices[train_end:]

# Build train and test sets.
X_train = Xn[train_ids]
y_train = yn[train_ids]

X_test = Xn[test_ids]
y_test = yn[test_ids]

# Check shapes before training.
assert X_train.shape[1] == 8
assert y_train.shape[1] == 1

# Choose a small hidden layer.
input_size = X_train.shape[1]
hidden_size = 6
output_size = 1

# Initialize weights and biases.
W1 = 0.5 * np.random.randn(input_size, hidden_size)
b1 = np.zeros((1, hidden_size))

W2 = 0.5 * np.random.randn(hidden_size, output_size)
b2 = np.zeros((1, output_size))

# Define the sigmoid activation.
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Define the sigmoid derivative.
def sigmoid_derivative(a):
    return a * (1.0 - a)

# Set simple training choices.
learning_rate = 0.5
epochs = 800

# Train the one hidden layer MLP.
for epoch in range(epochs):
    z1 = X_train @ W1 + b1
    a1 = sigmoid(z1)

    z2 = a1 @ W2 + b2
    y_pred = z2

    error = y_pred - y_train
    dW2 = (a1.T @ error) / X_train.shape[0]

    db2 = error.mean(axis=0, keepdims=True)
    hidden_error = (error @ W2.T) * sigmoid_derivative(a1)

    dW1 = (X_train.T @ hidden_error) / X_train.shape[0]
    db1 = hidden_error.mean(axis=0, keepdims=True)

    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2

    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1

# Run a forward pass on test data.
test_hidden = sigmoid(X_test @ W1 + b1)
test_pred = test_hidden @ W2 + b2

# Compute simple error measures.
test_mse = np.mean((test_pred - y_test) ** 2)
mae_norm = np.mean(np.abs(test_pred - y_test))

# Convert predictions back to MPa.
pred_strength = test_pred * y_range + y_min
true_strength = y_test * y_range + y_min

# Measure average MPa error.
mae_mpa = np.mean(np.abs(pred_strength - true_strength))

# Explain the layer structure clearly.
print("Input layer neurons:", input_size)
print("Hidden layer neurons:", hidden_size)
print("Output layer neurons:", output_size)
print("Train samples:", X_train.shape[0])
print("Test MSE normalized:", round(float(test_mse), 4))
print("Test MAE normalized:", round(float(mae_norm), 4))
print("Test MAE MPa:", round(float(mae_mpa), 2))
print("Example true normalized:", round(float(y_test[0, 0]), 3))
print("Example predicted normalized:", round(float(test_pred[0, 0]), 3))



### **2.3. Shallow and Deep Networks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_A/image_02_03.jpg?v=1774623014" width="250">



>* Depth means number of hidden layers.
>* More layers learn hierarchical, abstract features.

>* Shallow networks suit moderately complex relationships.
>* Deep networks learn layered, multiscale patterns.

>* Deeper networks add power but training costs.
>* Choose depth based on data structure.



In [ ]:
#@title Python Code - Shallow and Deep Networks

# This script compares shallow and deep networks.
# It uses concrete strength data.
# The focus is network structure differences.

# !pip install xlrd

# Import small required libraries.
import numpy as np
import pandas as pd
from pathlib import Path

# Set a deterministic random seed.
np.random.seed(7)
file_path = Path("Concrete_Data.xls")

# Download the dataset if missing.
if not file_path.exists():
    !wget -q -O ./Concrete_Data.xls https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls

data = pd.read_excel(file_path)

# Keep names short and beginner friendly.
X = data.iloc[:, :-1].to_numpy(dtype=float)
y = data.iloc[:, -1].to_numpy(dtype=float).reshape(-1, 1)

# Validate the loaded dataset shapes.
if X.shape[0] < 100 or X.shape[1] != 8:
    raise ValueError("Unexpected dataset shape.")

# Use a small subset for speed.
X = X[:700]
y = y[:700]

# Split into train and test parts.
cut = int(0.8 * len(X))
X_train = X[:cut]
X_test = X[cut:]

y_train = y[:cut]
y_test = y[cut:]

# Standardize inputs using training statistics.
X_mean = X_train.mean(axis=0, keepdims=True)
X_std = X_train.std(axis=0, keepdims=True) + 1e-8
X_train = (X_train - X_mean) / X_std
X_test = (X_test - X_mean) / X_std

# Standardize targets for stable learning.
y_mean = y_train.mean(axis=0, keepdims=True)
y_std = y_train.std(axis=0, keepdims=True) + 1e-8
y_train = (y_train - y_mean) / y_std
y_test = (y_test - y_mean) / y_std

# Define a simple tanh activation.
def tanh(x):
    return np.tanh(x)

# Define tanh derivative from outputs.
def tanh_grad(a):
    return 1.0 - a * a

# Build weights from layer sizes.
def make_network(sizes):
    params = []
    scale = 0.3
    i = 0

    while i < len(sizes) - 1:
        w = np.random.randn(sizes[i], sizes[i + 1]) * scale
        b = np.zeros((1, sizes[i + 1]))
        params.append([w, b])
        i = i + 1

    return params

# Count trainable parameters clearly.
def count_params(params):
    total = 0
    i = 0

    while i < len(params):
        w = params[i][0]
        b = params[i][1]
        total = total + w.size + b.size
        i = i + 1

    return total

# Run a forward pass.
def forward_pass(X_in, params):
    acts = [X_in]
    preacts = []
    i = 0

    while i < len(params):
        w = params[i][0]
        b = params[i][1]
        z = acts[-1] @ w + b
        preacts.append(z)

        if i < len(params) - 1:
            a = tanh(z)
        else:
            a = z

        acts.append(a)
        i = i + 1

    return acts, preacts

# Train with basic gradient descent.
def train_network(X_in, y_in, sizes, epochs, lr):
    params = make_network(sizes)
    n = X_in.shape[0]
    e = 0

    while e < epochs:
        acts, preacts = forward_pass(X_in, params)
        pred = acts[-1]
        delta = (pred - y_in) * (2.0 / n)
        i = len(params) - 1

        while i >= 0:
            a_prev = acts[i]
            grad_w = a_prev.T @ delta
            grad_b = delta.sum(axis=0, keepdims=True)
            params[i][0] = params[i][0] - lr * grad_w
            params[i][1] = params[i][1] - lr * grad_b

            if i > 0:
                back = delta @ params[i][0].T
                delta = back * tanh_grad(acts[i])

            i = i - 1

        e = e + 1

    return params

# Predict outputs from a network.
def predict(X_in, params):
    acts, preacts = forward_pass(X_in, params)
    return acts[-1]

# Compute mean squared error.
def mse(y_true, y_pred):
    diff = y_true - y_pred
    return float(np.mean(diff * diff))

# Define shallow and deep architectures.
shallow_sizes = [8, 10, 1]
deep_sizes = [8, 12, 8, 4, 1]

# Train both small demonstration networks.
shallow = train_network(X_train, y_train, shallow_sizes, 250, 0.03)
deep = train_network(X_train, y_train, deep_sizes, 250, 0.03)

# Evaluate both networks quietly.
shallow_pred = predict(X_test, shallow)
deep_pred = predict(X_test, deep)

shallow_mse = mse(y_test, shallow_pred)
deep_mse = mse(y_test, deep_pred)

# Print a short teaching summary.
print("Dataset rows used:", len(X))
print("Input features:", X.shape[1])
print("Shallow layers:", shallow_sizes)
print("Deep layers:", deep_sizes)
print("Shallow hidden layers:", len(shallow_sizes) - 2)
print("Deep hidden layers:", len(deep_sizes) - 2)
print("Shallow parameters:", count_params(shallow))
print("Deep parameters:", count_params(deep))
print("Shallow test MSE:", round(shallow_mse, 4))
print("Deep test MSE:", round(deep_mse, 4))
print("Depth adds more learned transformations.")



## **3. Neural Network Expressiveness**

### **3.1. Nonlinear Neuron Decisions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_A/image_03_01.jpg?v=1774623058" width="250">



>* Nonlinearity makes neural networks more expressive.
>* It captures changing civil engineering relationships.

>* Activation functions capture interacting input effects.
>* Neurons respond differently across operating regions.

>* Nonlinear neurons combine into complex predictive patterns.
>* They model intricate civil engineering relationships.



In [ ]:
#@title Python Code - Nonlinear Neuron Decisions

# Nonlinear neurons create flexible engineering decisions.
# This example uses satellite land cover data.
# One neuron shows a curved decision effect.

# Import simple learning tools.
import numpy as np
import pandas as pd

# Download the training file.
!wget -q -O ./sat.trn https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/satimage/sat.trn
# Download the testing file.
!wget -q -O ./sat.tst https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/satimage/sat.tst

# Set a deterministic random seed.
np.random.seed(7)
# Read both space separated files.
train_df = pd.read_csv("sat.trn", sep="\s+", header=None)
test_df = pd.read_csv("sat.tst", sep="\s+", header=None)

# Check that files loaded correctly.
if train_df.shape[1] != 37:
    raise ValueError("Training file should have 37 columns")
# Check that test file loaded correctly.
if test_df.shape[1] != 37:
    raise ValueError("Testing file should have 37 columns")

# Split features and labels.
X_train = train_df.iloc[:, :-1].to_numpy(dtype=float)
y_train_raw = train_df.iloc[:, -1].to_numpy(dtype=int)
X_test = test_df.iloc[:, :-1].to_numpy(dtype=float)
y_test_raw = test_df.iloc[:, -1].to_numpy(dtype=int)

# Build a binary site screening task.
# Class 4 means likely vegetation cover.
y_train = (y_train_raw == 4).astype(float)
y_test = (y_test_raw == 4).astype(float)

# Keep two features for easy explanation.
# These are two spectral measurements.
X_train_small = X_train[:, [0, 3]]
X_test_small = X_test[:, [0, 3]]

# Standardize using training statistics.
train_mean = X_train_small.mean(axis=0)
train_std = X_train_small.std(axis=0)
train_std[train_std == 0] = 1.0
X_train_small = (X_train_small - train_mean) / train_std
X_test_small = (X_test_small - train_mean) / train_std

# Add a nonlinear interaction feature.
# This helps one neuron bend decisions.
train_interaction = (X_train_small[:, 0] * X_train_small[:, 1]).reshape(-1, 1)
test_interaction = (X_test_small[:, 0] * X_test_small[:, 1]).reshape(-1, 1)
X_train_use = np.hstack([X_train_small, train_interaction])
X_test_use = np.hstack([X_test_small, test_interaction])

# Define the sigmoid activation.
def sigmoid(z):
    z = np.clip(z, -30, 30)
    return 1.0 / (1.0 + np.exp(-z))

# Start one neuron with small weights.
weights = np.zeros(X_train_use.shape[1])
bias = 0.0
learning_rate = 0.2
epochs = 120

# Train with simple gradient descent.
for epoch in range(epochs):
    scores = X_train_use @ weights + bias
    preds = sigmoid(scores)
    errors = preds - y_train
    grad_w = (X_train_use.T @ errors) / len(X_train_use)
    grad_b = errors.mean()

    # Update the neuron parameters.
    weights = weights - learning_rate * grad_w
    bias = bias - learning_rate * grad_b

# Make test predictions.
test_scores = X_test_use @ weights + bias
test_probs = sigmoid(test_scores)
test_pred = (test_probs >= 0.5).astype(int)

# Compute simple accuracy.
accuracy = (test_pred == y_test).mean()
# Count both classes for context.
positive_rate = y_test.mean()
# Show one neuron equation parts.
rounded_weights = np.round(weights, 3)

# Print a short teaching summary.
print("Binary task: vegetation versus other land cover")
print("Civil use: screen land near infrastructure corridors")
print("Train shape:", X_train_use.shape)
print("Test shape:", X_test_use.shape)
print("Positive class share:", round(float(positive_rate), 3))
print("Neuron weights:", rounded_weights)
print("Neuron bias:", round(float(bias), 3))
print("Test accuracy:", round(float(accuracy), 3))
print("Nonlinear term weight:", round(float(weights[2]), 3))
print("This interaction lets one neuron curve decisions.")



### **3.2. Layered Network Expressiveness**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_A/image_03_02.jpg?v=1774623096" width="250">



>* Layers build richer features from simple patterns.
>* This helps model complex civil engineering interactions.

>* Engineering effects develop through interacting stages.
>* Layers combine simple patterns into predictions.

>* Depth represents complex patterns more efficiently.
>* Layers match staged civil engineering processes.



In [ ]:
#@title Python Code - Layered Network Expressiveness

# This script shows layered network expressiveness.
# It uses air quality data.
# Deeper models capture richer nonlinear patterns.

import zipfile
import numpy as np
import pandas as pd

# Set seeds for repeatable results.
np.random.seed(7)

# Download the dataset file.
!wget -q -O ./AirQualityUCI.zip https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip

# Extract the downloaded archive.
with zipfile.ZipFile("./AirQualityUCI.zip", "r") as z:
    z.extractall("./air_quality_data")

# Load the main csv file.
file_path = "./air_quality_data/AirQualityUCI.csv"
df = pd.read_csv(file_path, sep=";", decimal=",")

# Remove empty trailing columns.
df = df.dropna(axis=1, how="all")

# Replace missing value markers.
df = df.replace(-200, np.nan)

# Keep useful sensor columns.
cols = ["PT08.S1(CO)", "NMHC(GT)", "C6H6(GT)", "PT08.S2(NMHC)", "T", "RH", "AH", "NO2(GT)"]
df = df[cols].dropna().copy()

# Limit rows for fast training.
df = df.iloc[:800].copy()

# Build a nonlinear target.
X = df[["T", "RH", "AH", "PT08.S1(CO)", "PT08.S2(NMHC)"]].to_numpy(dtype=float)
y = df["NO2(GT)"].to_numpy(dtype=float).reshape(-1, 1)

# Scale inputs and target.
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True) + 1e-8
X = (X - X_mean) / X_std

y_mean = y.mean(axis=0, keepdims=True)
y_std = y.std(axis=0, keepdims=True) + 1e-8
y = (y - y_mean) / y_std

# Add interaction features manually.
extra = np.column_stack((X[:, 0] * X[:, 1], X[:, 0] * X[:, 2], X[:, 3] * X[:, 4]))
X = np.column_stack((X, extra))

# Split into train and test sets.
n = len(X)
cut = int(0.8 * n)
X_train = X[:cut]
X_test = X[cut:]
y_train = y[:cut]
y_test = y[cut:]

# Check basic data shapes.
assert X_train.shape[0] > 50
assert X_train.shape[1] >= 5

# Define a simple activation.
def tanh(x):
    return np.tanh(x)

# Define activation derivative.
def tanh_grad(a):
    return 1.0 - a * a

# Train a single neuron model.
def train_single(X_data, y_data, steps, lr):
    w = 0.1 * np.random.randn(X_data.shape[1], 1)
    b = np.zeros((1, 1))

    for step in range(steps):
        pred = X_data @ w + b
        err = pred - y_data
        grad_w = (2.0 / len(X_data)) * (X_data.T @ err)
        grad_b = (2.0 / len(X_data)) * err.sum(axis=0, keepdims=True)

        w = w - lr * grad_w
        b = b - lr * grad_b

    return w, b

# Predict with single neuron.
def predict_single(X_data, w, b):
    return X_data @ w + b

# Train a two layer network.
def train_deep(X_data, y_data, hidden, steps, lr):
    W1 = 0.3 * np.random.randn(X_data.shape[1], hidden)
    b1 = np.zeros((1, hidden))
    W2 = 0.3 * np.random.randn(hidden, 1)
    b2 = np.zeros((1, 1))

    for step in range(steps):
        z1 = X_data @ W1 + b1
        a1 = tanh(z1)
        pred = a1 @ W2 + b2
        err = pred - y_data

        dW2 = (2.0 / len(X_data)) * (a1.T @ err)
        db2 = (2.0 / len(X_data)) * err.sum(axis=0, keepdims=True)
        da1 = err @ W2.T
        dz1 = da1 * tanh_grad(a1)

        dW1 = (2.0 / len(X_data)) * (X_data.T @ dz1)
        db1 = (2.0 / len(X_data)) * dz1.sum(axis=0, keepdims=True)
        W2 = W2 - lr * dW2
        b2 = b2 - lr * db2

        W1 = W1 - lr * dW1
        b1 = b1 - lr * db1

    return W1, b1, W2, b2

# Predict with deep network.
def predict_deep(X_data, W1, b1, W2, b2):
    hidden = tanh(X_data @ W1 + b1)
    return hidden @ W2 + b2

# Compute mean squared error.
def mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred) ** 2))

# Compute simple correlation value.
def corr_value(y_true, y_pred):
    a = y_true.ravel()
    b = y_pred.ravel()
    return float(np.corrcoef(a, b)[0, 1])

# Fit both models quietly.
w, b = train_single(X_train, y_train, 400, 0.03)
W1, b1, W2, b2 = train_deep(X_train, y_train, 8, 600, 0.03)

# Make test predictions.
pred_single = predict_single(X_test, w, b)
pred_deep = predict_deep(X_test, W1, b1, W2, b2)

# Measure model quality.
mse_single = mse(y_test, pred_single)
mse_deep = mse(y_test, pred_deep)
cor_single = corr_value(y_test, pred_single)
cor_deep = corr_value(y_test, pred_deep)

# Print a short teaching summary.
print("Rows used:", len(df), " Test rows:", len(X_test))
print("Target: standardized NO2 estimation")
print("Single neuron test MSE:", round(mse_single, 4))
print("Two layer network test MSE:", round(mse_deep, 4))
print("Single neuron correlation:", round(cor_single, 4))
print("Two layer correlation:", round(cor_deep, 4))
print("Layered model learns combined nonlinear effects better.")



### **3.3. Approximation Power**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_A/image_03_03.jpg?v=1774623140" width="250">



>* Neural networks approximate complex nonlinear functions.
>* Useful for modeling complex civil systems.

>* Engineering relationships involve many interacting factors.
>* Neural networks model complex patterns from data.

>* Approximation shows potential, not guaranteed success.
>* Good data and validation remain essential.



In [ ]:
#@title Python Code - Approximation Power

# Neural networks can approximate complex engineering relationships.
# This example compares linear and nonlinear predictors.
# Concrete strength data shows approximation power clearly.

# Install Excel reader for pandas.
# !pip install xlrd

# Import beginner friendly tools.
import numpy as np
import pandas as pd

# Set a deterministic random seed.
np.random.seed(7)

# Download the concrete dataset.
!wget -q -O ./Concrete_Data.xls https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls

# Load the spreadsheet into pandas.
df = pd.read_excel("./Concrete_Data.xls")

# Clean column names for convenience.
df.columns = [str(c).strip() for c in df.columns]

# Keep a manageable sample size.
df = df.sample(n=500, random_state=7)

# Convert data into numpy arrays.
X = df.iloc[:, :-1].to_numpy(dtype=float)
y = df.iloc[:, -1].to_numpy(dtype=float).reshape(-1, 1)

# Check that shapes look valid.
assert X.shape[0] == y.shape[0]

# Split data into train and test.
idx = np.arange(X.shape[0])
np.random.shuffle(idx)

# Use eighty percent for training.
split = int(0.8 * len(idx))
train_idx = idx[:split]
test_idx = idx[split:]

# Create train and test arrays.
X_train = X[train_idx]
X_test = X[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

# Standardize inputs using training statistics.
X_mean = X_train.mean(axis=0, keepdims=True)
X_std = X_train.std(axis=0, keepdims=True) + 1e-8

# Apply the same scaling everywhere.
X_train_s = (X_train - X_mean) / X_std
X_test_s = (X_test - X_mean) / X_std

# Standardize targets for stable training.
y_mean = y_train.mean(axis=0, keepdims=True)
y_std = y_train.std(axis=0, keepdims=True) + 1e-8

y_train_s = (y_train - y_mean) / y_std
y_test_s = (y_test - y_mean) / y_std

# Add a bias column manually.
ones_train = np.ones((X_train_s.shape[0], 1))
ones_test = np.ones((X_test_s.shape[0], 1))

# Build linear design matrices.
Xb_train = np.hstack((X_train_s, ones_train))
Xb_test = np.hstack((X_test_s, ones_test))

# Fit linear regression by least squares.
linear_w = np.linalg.pinv(Xb_train) @ y_train_s

# Predict with the linear model.
linear_pred_s = Xb_test @ linear_w
linear_pred = linear_pred_s * y_std + y_mean

# Define a simple relu function.
def relu(z):
    return np.maximum(0.0, z)

# Define relu derivative values.
def relu_grad(z):
    return (z > 0).astype(float)

# Choose a small hidden layer.
input_size = X_train_s.shape[1]
hidden_size = 12

# Initialize network weights safely.
W1 = 0.3 * np.random.randn(input_size, hidden_size)
b1 = np.zeros((1, hidden_size))

# Initialize output layer weights.
W2 = 0.3 * np.random.randn(hidden_size, 1)
b2 = np.zeros((1, 1))

# Set training hyperparameters.
learning_rate = 0.03
epochs = 800

# Train the hand coded network.
for epoch in range(epochs):
    Z1 = X_train_s @ W1 + b1
    A1 = relu(Z1)

    Z2 = A1 @ W2 + b2
    pred = Z2

    error = pred - y_train_s
    dZ2 = (2.0 / X_train_s.shape[0]) * error

    dW2 = A1.T @ dZ2
    db2 = dZ2.sum(axis=0, keepdims=True)

    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * relu_grad(Z1)

    dW1 = X_train_s.T @ dZ1
    db1 = dZ1.sum(axis=0, keepdims=True)

    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2

    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1

# Predict with the neural network.
Z1_test = X_test_s @ W1 + b1
A1_test = relu(Z1_test)

# Convert predictions back to MPa.
nn_pred_s = A1_test @ W2 + b2
nn_pred = nn_pred_s * y_std + y_mean

# Define a simple error metric.
def rmse(y_true, y_hat):
    return float(np.sqrt(np.mean((y_true - y_hat) ** 2)))

# Compute test errors for both models.
linear_rmse = rmse(y_test, linear_pred)
nn_rmse = rmse(y_test, nn_pred)

# Show a few example predictions.
show_n = 3
actual_vals = y_test[:show_n, 0]
linear_vals = linear_pred[:show_n, 0]

# Prepare neural network examples.
nn_vals = nn_pred[:show_n, 0]

# Print concise teaching results.
print("Dataset rows used:", X.shape[0])
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Linear RMSE MPa:", round(linear_rmse, 2))
print("Neural net RMSE MPa:", round(nn_rmse, 2))
print("Lower RMSE means better approximation.")
print("Example actual MPa:", np.round(actual_vals, 1))
print("Linear predictions:", np.round(linear_vals, 1))
print("Neural predictions:", np.round(nn_vals, 1))
print("Hidden neurons let the network fit nonlinear patterns.")



# <font color="#418FDE" size="6.5" uppercase>**Neural Network Foundations**</font>


In this lecture, you learned to:
- Explain the perceptron model, its biological inspiration, and the roles of weights, biases, and activation functions. 
- Describe the structure of multi-layer perceptrons and distinguish between shallow and deep neural networks. 
- Interpret the significance of the universal approximation theorem for modeling complex CE relationships. 

In the next Lecture (Lecture B), we will go over 'Neural Network Optimization and Training'